# Citi Bike Trip Data: 16 Pandas Exercises

**Dataset:** [Official NYC Citi Bike Trip Histories](https://citibikenyc.com/system-data) · [Monthly CSV archives](https://s3.amazonaws.com/tripdata/index.html)

This notebook uses the **modern 13-column trip schema** (ride_id, rideable_type, started_at, ended_at, start/end station IDs and names, coordinates, and member_casual). Download **one NYC month** (not a Jersey City archive prefixed `JC`) and place its `.zip` archive or extracted `.csv` file(s) in a `data/` folder next to this notebook. One monthly ZIP can contain multiple CSVs, which will all be read.

Run cells top to bottom. Each question has a worked Python solution directly beneath it. Visualizations use **Matplotlib** and **Plotly**. The data are not included with this notebook, so results will depend on the chosen month.

## Setup: Install packages and load one month's data

Create `data/` and place one modern Citi Bike monthly `.zip` archive **or** its extracted `.csv` files there. If using extracted files, avoid mixing months unless deliberately performing a multi-month analysis. The cell detects input automatically. If memory is limited, set `MAX_ROWS` to an integer (for example `300_000`), but this reads the first rows rather than a representative random sample and may bias time-of-day findings.

In [ ]:
# Uncomment once if needed:
# %pip install pandas numpy matplotlib plotly

from pathlib import Path
from zipfile import ZipFile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.express as px

DATA_DIR = Path("data")
MAX_ROWS = None  # None = full dataset; integer = first N rows (nonrandom sample)

zip_files = sorted(DATA_DIR.glob("*.zip"))
csv_files = sorted(DATA_DIR.glob("*.csv"))
if zip_files and csv_files:
    raise ValueError("Use ZIP archives OR extracted CSVs in data/, not both (avoid double-counting).")
if not zip_files and not csv_files:
    raise FileNotFoundError("Download a Citi Bike monthly ZIP/CSV and put it in data/.")

pieces = []
if zip_files:
    for archive in zip_files:
        with ZipFile(archive) as z:
            for name in sorted(z.namelist()):
                if name.lower().endswith('.csv') and not name.startswith('__MACOSX'):
                    with z.open(name) as f:
                        pieces.append(pd.read_csv(f, nrows=MAX_ROWS, low_memory=False,
                                                  dtype={'start_station_id':'string', 'end_station_id':'string'}))
else:
    for file in csv_files:
        pieces.append(pd.read_csv(file, nrows=MAX_ROWS, low_memory=False,
                                  dtype={'start_station_id':'string', 'end_station_id':'string'}))

df = pd.concat(pieces, ignore_index=True)
required = {'ride_id','rideable_type','started_at','ended_at','start_station_name',
            'start_station_id','end_station_name','end_station_id',
            'start_lat','start_lng','end_lat','end_lng','member_casual'}
missing_columns = required - set(df.columns)
if missing_columns:
    raise ValueError(f"Not a modern Citi Bike trip file; missing: {sorted(missing_columns)}")

for col in ('started_at','ended_at'):
    df[col] = pd.to_datetime(df[col], errors='coerce')
for col in ('start_lat','start_lng','end_lat','end_lng'):
    df[col] = pd.to_numeric(df[col], errors='coerce')

print(f'{len(df):,} rows from {len(pieces)} CSV file(s)')
print('Date range:', df.started_at.min(), 'to', df.started_at.max())
df.head()

## Exercise 1 — Getting acquainted with the data

1. How many trip records are there? How many unique start stations and end stations?
2. List the observed bike types and rider categories and their counts.
3. Identify missing fields and duplicate `ride_id` values. Which missing data could affect later analyses?

In [ ]:
print('Shape:', df.shape)
display(df.info(), df.describe(include='all').T)
print('Unique start stations:', df.start_station_id.nunique())
print('Unique end stations:', df.end_station_id.nunique())
print('Bike types:\n', df.rideable_type.value_counts(dropna=False))
print('Rider categories:\n', df.member_casual.value_counts(dropna=False))
print('Missing values:\n', df.isna().sum().sort_values(ascending=False))
print('Duplicate ride IDs:', df.ride_id.duplicated().sum())

### Shared cleaning and derived variables

All subsequent exercises use `rides`: records with valid trip start/end times and **positive durations**. We retain valid but unusually long trips for investigation rather than silently deleting them. Station-based analyses drop missing station IDs only when needed. A weekday is Monday–Friday.

In [ ]:
rides = df.drop_duplicates(subset='ride_id').copy()
rides['duration_min'] = (rides.ended_at - rides.started_at).dt.total_seconds() / 60
rides = rides.loc[rides.started_at.notna() & rides.ended_at.notna() &
                  rides.duration_min.gt(0)].copy()
rides['hour'] = rides.started_at.dt.hour
rides['weekday_num'] = rides.started_at.dt.dayofweek  # Monday=0
rides['weekday'] = pd.Categorical(
    rides.started_at.dt.day_name(),
    categories=['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday'],
    ordered=True)
rides['is_weekend'] = rides.weekday_num.ge(5)
rides['date'] = rides.started_at.dt.date
print('Valid trips retained:', f'{len(rides):,}', 'of', f'{len(df):,}')

## Exercise 2 — How long do people ride?

1. Compute trip duration in minutes from the two timestamps.
2. Report the mean, median, standard deviation, and 10th, 90th, and 99th percentiles.
3. Count trips shorter than 2 minutes and longer than 2 hours; visualize the distribution up to the 99th percentile. Explain why mean and median may disagree.

## Exercise 3 — Do members ride differently?

Compare members and casual riders using counts, mean and median duration, and 90th percentile duration. Create side-by-side boxplots with extreme trips excluded **only for visualization**. Which summary statistic is more robust?

## Exercise 4 — When are bikes used?

Count trip starts by hour and day of week. Produce a **7 × 24** heatmap with weekdays ordered Monday through Sunday. Which day/hour combinations are busiest?

## Exercise 5 — Commuters versus recreational riders

For each rider category, compare hourly trip profiles separately for weekdays and weekends. **Normalize by the number of observed weekdays/weekend days** so the comparison isn't merely driven by there being more weekdays. What patterns suggest commuting?

## Exercise 6 — Are electric bikes used differently?

Compare the count and median duration of each bike type, broken down by rider category. Then show how each bike type's **share of trips** changes through the day (not just its raw trip count).

## Exercise 7 — Which stations are busiest?

Find the 20 stations with the most departures, arrivals, and total traffic. Compare rankings and discuss why a station might rank differently depending on the measure. Use **station IDs**, not names, as grouping keys.

In [ ]:
departures = rides.dropna(subset=['start_station_id']).groupby('start_station_id').size().rename('departures')
arrivals = rides.dropna(subset=['end_station_id']).groupby('end_station_id').size().rename('arrivals')
station_names = pd.concat([
    rides[['start_station_id','start_station_name']].rename(columns={'start_station_id':'station_id','start_station_name':'name'}),
    rides[['end_station_id','end_station_name']].rename(columns={'end_station_id':'station_id','end_station_name':'name'})
]).dropna(subset=['station_id']).drop_duplicates('station_id').set_index('station_id')['name']
station_traffic = pd.concat([departures, arrivals], axis=1).fillna(0).astype(int)
station_traffic['total'] = station_traffic.departures + station_traffic.arrivals
station_traffic['station_name'] = station_names.reindex(station_traffic.index)
for metric in ['departures','arrivals','total']:
    print(f'\nTop 20 by {metric}')
    display(station_traffic.nlargest(20,metric)[['station_name',metric]])

## Exercise 8 — Where do bikes accumulate?

Calculate each station's **net inflow** = arrivals minus departures, and list the ten largest gains and losses. Explain why these are not direct measurements of dock inventory or fleet redistribution.

## Exercise 9 — What are the busiest routes?

Identify the top 20 ordered start → end station pairs and separately identify the top 10 routes for weekdays and weekends. **A→B and B→A are different routes.**

## Exercise 10 — Do bikes return to their starting station?

What percent of trips have the same start and end station ID? Compare the proportions across member/casual riders and bike types. Exclude trips with unknown station IDs to avoid treating two missing values as a match.

## Exercise 11 — Map: Where are the busiest stations?

Make an interactive map of the **100 stations with the most departures**. Scale marker sizes by departure count and color them by the percentage of departing riders who are casual. Add station names to hover labels. Describe any geographic clusters.

**Coordinate caution:** Trip files store trip endpoints, which may differ slightly from official dock coordinates. We approximate station coordinates using median trip start coordinates grouped by station ID. Invalid geographic coordinates are excluded.

## Exercise 12 — Map: Can station imbalances reveal redistribution needs?

For each station, calculate net inflow (arrivals minus departures) **during weekday mornings (7–10)** and **weekday evenings (16–19)**. Map the morning net inflow with diverging colors (net loss versus gain) and marker size proportional to absolute imbalance. Compare it to the evening map. Which stations reverse direction?

These metrics show observed trip flows only—not actual docking capacity, availability, or redistribution operations.

In [ ]:
# Create period-specific station flows, combining outbound and inbound station IDs.
weekdays = rides[~rides.is_weekend].copy()
windows = {'Morning (07–10)': (7,10), 'Evening (16–19)': (16,19)}
period_tables = {}
for label, (start_hour, stop_hour) in windows.items():
    part = weekdays[weekdays.hour.between(start_hour,stop_hour)]
    out = part.groupby('start_station_id').size().rename('out')
    inbound = part.groupby('end_station_id').size().rename('in')
    flow = pd.concat([out,inbound],axis=1).fillna(0)
    flow['net'] = flow['in'] - flow['out']
    flow.index.name = 'station_id'
    period_tables[label] = flow

map_coords = station_map.rename(columns={'start_station_id':'station_id'})
for label, flow in period_tables.items():
    m = map_coords.merge(flow[['net']], left_on='station_id', right_index=True, how='inner')
    m['abs_net'] = m.net.abs().clip(lower=1)
    fig = px.scatter_mapbox(m, lat='lat', lon='lon', size='abs_net', color='net',
        hover_name='station_name', hover_data={'net':True,'abs_net':False,'lat':False,'lon':False},
        color_continuous_scale='RdBu', color_continuous_midpoint=0,
        size_max=23, zoom=10, title=f'Weekday {label}: net inflow')
    fig.update_layout(mapbox_style='open-street-map',margin=dict(l=0,r=0,t=45,b=0))
    fig.show()

morning = period_tables['Morning (07–10)']['net'].rename('morning_net')
evening = period_tables['Evening (16–19)']['net'].rename('evening_net')
reversals = pd.concat([morning,evening],axis=1).dropna()
reversals = reversals[reversals.morning_net * reversals.evening_net < 0]
reversals['combined_abs_flow'] = reversals.morning_net.abs() + reversals.evening_net.abs()
display(reversals.nlargest(15,'combined_abs_flow'))

## Exercise 13 — How far do people travel? (Challenge)

Estimate each trip's **great-circle (straight-line) distance** using the Haversine formula, then compare distances and trip durations by bike type. Why isn't this an estimate of actual route mileage? How should same-station trips be interpreted?

In [ ]:
distance_rides = rides.dropna(subset=['start_lat','start_lng','end_lat','end_lng']).copy()
distance_rides = distance_rides[
    distance_rides.start_lat.between(40.3,41.2) & distance_rides.end_lat.between(40.3,41.2) &
    distance_rides.start_lng.between(-74.5,-73.3) & distance_rides.end_lng.between(-74.5,-73.3)]
R = 6371.0  # Earth mean radius, km
lat1, lon1 = np.radians(distance_rides.start_lat), np.radians(distance_rides.start_lng)
lat2, lon2 = np.radians(distance_rides.end_lat), np.radians(distance_rides.end_lng)
dlat, dlon = lat2-lat1, lon2-lon1
a = np.sin(dlat/2)**2 + np.cos(lat1)*np.cos(lat2)*np.sin(dlon/2)**2
distance_rides['straight_km'] = 2*R*np.arcsin(np.sqrt(np.clip(a,0,1)))
display(distance_rides.groupby('rideable_type')[['straight_km','duration_min']].agg(['count','median','mean']))

## Exercise 14 — Which stations are commuter hubs? (Challenge)

Develop and explain a **commuter score** from weekday morning net departures, weekday evening net arrivals, and weekday-versus-weekend relative trip volume. Rank ten likely residential-origin commuter stations. This is a *heuristic*, not a validated commuter label.

## Exercise 15 — Can we predict hourly station demand? (Challenge)

Choose the busiest departure station. Generate a continuous hourly time series (including zero-trip hours), then forecast the final 20% of hours using the **average from the same hour of week** in the training data. Compare against a naive constant-mean forecast using mean absolute error (MAE). 

For strongest comparisons, use at least several weeks of unsampled data.

## Exercise 16 — How concentrated is ridership? (Challenge)

Rank stations by departures, calculate what share of trips begins at the busiest **5%, 10%, and 20% of stations**, and plot the cumulative percentage of trips versus the cumulative percentage of stations. Discuss the operational implications of concentration.

## Interpretation and limitations (all exercises)

For each exercise, submit a short interpretation alongside code/output. Address **missing station IDs**, **outlier durations**, **data period and sampling**, **station coordinate approximation**, and the distinction between **trip flows** and **actual dock inventory**. Straight-line distance is not traveled distance; peak trip counts are not proof of unmet demand. When comparing weekdays with weekends, normalize for observation period.